# Task 3 - Domain Generalization

PACS ERM reuse, source-only alignment, and SAM without Sketch access before final evaluation.

**Execution policy:** this notebook is intentionally delivered unexecuted. Set the configuration paths and switches, then run top-to-bottom when you are ready to conduct the experiment. It does not answer the report questions.

## 1. Configuration and strict unseen-target protocol

No Sketch files are loaded until the explicitly marked final-evaluation section. This notebook reuses Task 2's fixed source split and Source-only checkpoint.

The next cell records source-only configurations and paths. Sketch is deliberately absent from the configuration and all subsequent training cells.

In Colab, this configuration mounts Drive. Large inputs and checkpoints live under `ATML_PA1/` in Drive; small final outputs go to this task's repository `results/` directory. The repository name is detected automatically.

In [ ]:
# Standard library and experiment dependencies.
# Install dependencies yourself before executing: torch torchvision open_clip_torch
# scikit-learn pandas matplotlib seaborn pillow scipy tqdm
import json, random, math, copy
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from torchvision.transforms import InterpolationMode

SEED = 6304
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


### Repository and Drive paths

The next cell locates the clone, mounts Drive in Colab, and creates persistent data/checkpoint/artifact directories plus the task's small-results directory.

In [ ]:

# The clone holds code and small final outputs; Drive retains large inputs and models.
REPO_ROOT = Path('/content/atml-pa1')
TASK_ROOT = REPO_ROOT / "task3"
REPO_RESULTS_DIR = TASK_ROOT / "results"

import os

if "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import drive

    drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/ATML_PA1")
DATA_ROOT = DRIVE_ROOT / "data"
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / "task3"
ARTIFACT_DIR = DRIVE_ROOT / "artifacts" / "task3"
EXTERNAL_DIR = DRIVE_ROOT / "external"
TORCH_CACHE_DIR = EXTERNAL_DIR / "torch_cache"
HF_CACHE_DIR = EXTERNAL_DIR / "huggingface_cache"

for directory in (
    REPO_RESULTS_DIR,
    DATA_ROOT,
    CHECKPOINT_DIR,
    ARTIFACT_DIR,
    EXTERNAL_DIR,
    TORCH_CACHE_DIR,
    HF_CACHE_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

# Pretrained-weight downloads also survive a Colab runtime reset.
os.environ["TORCH_HOME"] = str(TORCH_CACHE_DIR)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)


### Reproducibility and display helpers

The next cell defines the fixed-seed behavior and small output helpers. It does not run an experiment.

In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def save_json(value, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(value, f, indent=2)


def show_table(rows, title=None):
    frame = pd.DataFrame(rows)
    if title:
        print(title)
    display(frame)
    return frame


set_seed()


### Task settings

The next cell records this task's data path, fixed experiment settings, and plot/print_metrics switches. It produces no metrics.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

CONFIG = {
    "pacs_root": DATA_ROOT / "PACS",
    "split_file": REPO_ROOT / "common" / "splits" / "pacs_sketch_seed6304.json",
    "erm_checkpoint": DRIVE_ROOT / "checkpoints" / "task2" / "source_only.pt",
    "results_dir": REPO_RESULTS_DIR,
    "epochs": 30,
    "patience": 5,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "batch_per_domain": 8,
    "lambda_dg": 1.0,
    "rho": 0.05,
    "study_rhos": [0.01, 0.05, 0.1],
    "plot": True,
    "print_metrics": True,
}
SOURCES = ["photo", "art_painting", "cartoon"]
CLASSES = ["dog", "elephant", "giraffe", "guitar", "horse", "house", "person"]
RESULTS = Path(CONFIG["results_dir"])
RESULTS.mkdir(parents=True, exist_ok=True)


## 2. Source-only data utilities (Sketch deliberately absent)

The next cell loads only Photo, Art Painting, and Cartoon plus the saved Task 2 split. It defines preprocessing, the ResNet-18 feature path, MMD, and source-side evaluation.

In [ ]:
weights = ResNet18_Weights.IMAGENET1K_V1
train_tf = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.RandomCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(weights.transforms().mean, weights.transforms().std),
    ]
)
eval_tf = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(weights.transforms().mean, weights.transforms().std),
    ]
)


# Only the three observed source directories are enumerated in this function.
def source_index():
    records = []
    for d in SOURCES:
        if not (CONFIG["pacs_root"] / d).is_dir():
            raise FileNotFoundError(
                f"PACS source domain is missing from Google Drive: {CONFIG['pacs_root'] / d}"
            )
        for y, c in enumerate(CLASSES):
            class_dir = CONFIG["pacs_root"] / d / c
            for image_path in sorted(class_dir.glob("*")):
                records.append(
                    {"path": str(image_path), "domain": d, "label": y}
                )
    frame = pd.DataFrame(records)
    if not CONFIG["split_file"].is_file():
        raise FileNotFoundError(
            f"Task 2 source split is missing from the cloned repository: {CONFIG['split_file']}"
        )
    saved = pd.DataFrame(json.load(open(CONFIG["split_file"])))
    return frame.merge(saved[["path", "split"]], on="path", how="left", validate="one_to_one")


source = source_index()


class PACS(Dataset):
    def __init__(self, frame, tf):
        self.frame = frame.reset_index(drop=True)
        self.tf = tf

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, i):
        r = self.frame.iloc[i]
        return self.tf(Image.open(r.path).convert("RGB")), int(r.label), r.domain


def freeze_bn(m):
    for module in m.modules():
        if isinstance(module, nn.modules.batchnorm._BatchNorm):
            module.eval()


def make_model():
    m = resnet18(weights=weights)
    m.fc = nn.Linear(512, 7)
    return m.to(DEVICE)


def outputs(m, x):
    # Retain the 512-D pooled feature before the seven-class classifier.
    stem = m.relu(m.bn1(m.conv1(x)))
    h = m.maxpool(stem)
    h = m.layer1(h)
    h = m.layer2(h)
    h = m.layer3(h)
    h = m.layer4(h)
    f = torch.flatten(m.avgpool(h), 1)
    return f, m.fc(f)


### Source-domain alignment and evaluation utilities

The following functions define the source-only MMD term, domain-balanced loaders, and validation metrics. They do not enumerate Sketch.

In [ ]:
# DAN-DG averages marginal MMD over source-domain pairs; Sketch is not an input.
def mmd(a, b):
    joined = torch.cat([a, b])
    d = torch.cdist(joined, joined).square()
    med = d[d > 0].median().detach().clamp_min(1e-6)
    # The three fixed kernel scales match the Task 2 alignment definition.
    k = 0
    for scale in (0.5, 1, 2):
        k = k + torch.exp(-d / (2 * scale * med))
    n = len(a)
    return k[:n, :n].mean() + k[n:, n:].mean() - 2 * k[:n, n:].mean()


def loaders(split, tf):
    result = {}
    for domain in SOURCES:
        selected = source[(source.domain == domain) & (source.split == split)]
        dataset = PACS(selected, tf)
        result[domain] = DataLoader(
            dataset,
            batch_size=CONFIG["batch_per_domain"],
            shuffle=True,
            drop_last=True,
        )
    return result


def evaluate(m, frame):
    m.eval()
    freeze_bn(m)
    ys = []
    ps = []
    for x, y, _ in DataLoader(PACS(frame, eval_tf), batch_size=64):
        with torch.no_grad():
            _, z = outputs(m, x.to(DEVICE))
            ps.extend(z.argmax(1).cpu())
            ys.extend(y)
    return {
        "accuracy": accuracy_score(ys, ps),
        "macro_f1": f1_score(ys, ps, average="macro"),
        "y": np.array(ys),
        "pred": np.array(ps),
    }


## 3. ERM checkpoint reuse, DAN-DG, and SAM training

The next cell loads the Task 2 ERM checkpoint and trains DAN-DG and SAM. It records source-side classification and MMD curves, selecting checkpoints only from source-validation macro-F1.

In [ ]:
if not CONFIG["erm_checkpoint"].is_file():
    raise FileNotFoundError(
        f"Run Task 2 first; its selected Source-only checkpoint is missing from Google Drive: {CONFIG['erm_checkpoint']}"
    )
erm = make_model()
erm.load_state_dict(torch.load(CONFIG["erm_checkpoint"], map_location=DEVICE)["model"])


# SAM requires two passes: find a local loss-increasing perturbation, then update at that point.
def train(method, rho=CONFIG["rho"]):
    m = make_model()
    opt = torch.optim.AdamW(m.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
    its = {}
    for domain, loader in loaders("train", train_tf).items():
        its[domain] = iter(loader)
    best = -1
    stale = 0
    state = None
    history = []
    for epoch in range(CONFIG["epochs"]):
        m.train()
        freeze_bn(m)
        total_cls = total_mmd = 0
        # Cover the longest source domain and cycle the shorter source loaders.
        steps = max(map(len, loaders("train", train_tf).values()))
        for _ in range(steps):
            batches = []
            for d in SOURCES:
                try:
                    b = next(its[d])
                except StopIteration:
                    its[d] = iter(loaders("train", train_tf)[d])
                    b = next(its[d])
                batches.append(b)
            x = torch.cat([b[0] for b in batches]).to(DEVICE)
            y = torch.cat([b[1] for b in batches]).to(DEVICE)
            f, z = outputs(m, x)
            cls = F.cross_entropy(z, y)
            # Each contiguous feature block is one source domain: [8, 512].
            penalty = torch.zeros((), device=DEVICE)
            if method == "dan_dg":
                pairwise_mmd = 0
                for first, second in [(0, 1), (0, 2), (1, 2)]:
                    first_features = f[first * 8 : (first + 1) * 8]
                    second_features = f[second * 8 : (second + 1) * 8]
                    pairwise_mmd += mmd(first_features, second_features)
                penalty = pairwise_mmd / 3
            loss = cls + CONFIG["lambda_dg"] * penalty
            if method == "sam":
                opt.zero_grad()
                loss.backward()
                # SAM normalizes the ascent step across all trainable gradients.
                squared_gradients = 0
                for parameter in m.parameters():
                    if parameter.grad is not None:
                        squared_gradients += (parameter.grad**2).sum()
                norm = torch.sqrt(squared_gradients)
                perturb = []
                with torch.no_grad():
                    for p in m.parameters():
                        e = (
                            rho * p.grad / (norm + 1e-12)
                            if p.grad is not None
                            else torch.zeros_like(p)
                        )
                        p.add_(e)
                        perturb.append(e)
                freeze_bn(m)
                _, z2 = outputs(m, x)
                opt.zero_grad()
                F.cross_entropy(z2, y).backward()
                with torch.no_grad():
                    for p, e in zip(m.parameters(), perturb):
                        p.sub_(e)
                opt.step()
            else:
                opt.zero_grad()
                loss.backward()
                opt.step()
            total_cls += cls.item()
            total_mmd += penalty.item()
        source_f1_values = []
        for domain in SOURCES:
            validation_rows = source[
                (source.domain == domain) & (source.split == "val")
            ]
            validation_metrics = evaluate(m, validation_rows)
            source_f1_values.append(validation_metrics["macro_f1"])
        # Sketch never contributes to model selection.
        score = np.mean(source_f1_values)
        history.append(
            {
                "epoch": epoch + 1,
                "classification_loss": total_cls / steps,
                "mmd_penalty": total_mmd / steps,
                "mean_source_f1": score,
            }
        )
        if score > best:
            best = score
            stale = 0
            state = copy.deepcopy(m.state_dict())
        else:
            stale += 1
        if stale >= CONFIG["patience"]:
            break
    m.load_state_dict(state)
    return m, history


### Train DAN-DG and SAM

ERM above is loaded from Task 2 rather than retrained. This cell trains the two Task 3 methods in the existing order and keeps their selected states and histories in memory.

In [ ]:
dan_dg, dan_history = train("dan_dg")
sam, sam_history = train("sam")


## 4. Source diagnostics only: metrics, source-domain separability, and sharpness proxy

The next cell computes source-domain separability and the common sharpness proxy using only source validation data, then saves the source diagnostic table.

In [ ]:
# Chance is 33.3% because this probe predicts one of three source domains.
def source_separability(m):
    # Balance the probe across all three observed domains before splitting 70/30.
    validation_groups = [source[(source.domain == d) & (source.split == "val")] for d in SOURCES]
    per_domain = min(len(group) for group in validation_groups)
    balanced_groups = []
    for domain_index, group in enumerate(validation_groups):
        balanced_groups.append(
            group.sample(per_domain, random_state=SEED).assign(domain_id=domain_index)
        )
    frame = pd.concat(balanced_groups)
    fs = []
    ds = []
    for x, _, d in DataLoader(PACS(frame, eval_tf), batch_size=64):
        with torch.no_grad():
            fs.append(outputs(m, x.to(DEVICE))[0].cpu())
            ds += [SOURCES.index(v) for v in d]
    ids = np.random.default_rng(SEED).permutation(len(ds))
    cut = int(0.7 * len(ids))
    clf = LogisticRegression(C=1, max_iter=2000, multi_class="auto").fit(
        torch.cat(fs)[ids[:cut]], np.array(ds)[ids[:cut]]
    )
    return clf.score(torch.cat(fs)[ids[cut:]], np.array(ds)[ids[cut:]])


def sharpness(m):
    fixed = pd.concat(
        [
            source[(source.domain == d) & (source.split == "val")].sample(32, random_state=SEED)
            for d in SOURCES
        ]
    )
    x, y, _ = next(iter(DataLoader(PACS(fixed, eval_tf), batch_size=96)))
    m.eval()
    freeze_bn(m)
    _, z = outputs(m, x.to(DEVICE))
    base = F.cross_entropy(z, y.to(DEVICE))
    m.zero_grad()
    base.backward()
    norm = torch.sqrt(sum((p.grad**2).sum() for p in m.parameters() if p.grad is not None))
    es = []
    with torch.no_grad():
        for p in m.parameters():
            e = 0.05 * p.grad / (norm + 1e-12) if p.grad is not None else torch.zeros_like(p)
            p.add_(e)
            es.append(e)
    with torch.no_grad():
        _, z = outputs(m, x.to(DEVICE))
        perturbed = F.cross_entropy(z, y.to(DEVICE))
    with torch.no_grad():
        for p, e in zip(m.parameters(), es):
            p.sub_(e)
    return (perturbed - base).item()


### Source-only diagnostics

Only source validation images enter these metrics. The next cell saves the source diagnostic table before any Sketch file is opened.

In [ ]:
source_rows = []
for name, m in {"erm": erm, "dan_dg": dan_dg, "sam": sam}.items():
    scores = {
        d: evaluate(m, source[(source.domain == d) & (source.split == "val")]) for d in SOURCES
    }
    source_rows.append(
        {
            "method": name,
            **{f"{d}_accuracy": scores[d]["accuracy"] for d in SOURCES},
            **{f"{d}_macro_f1": scores[d]["macro_f1"] for d in SOURCES},
            "mean_source_accuracy": np.mean([s["accuracy"] for s in scores.values()]),
            "worst_source_accuracy": min(s["accuracy"] for s in scores.values()),
            "mean_source_macro_f1": np.mean([s["macro_f1"] for s in scores.values()]),
            "worst_source_macro_f1": min(s["macro_f1"] for s in scores.values()),
            "source_domain_separability": source_separability(m),
            "sharpness_proxy": sharpness(m),
        }
    )
source_frame = pd.DataFrame(source_rows)
source_frame.to_csv(RESULTS / "source_diagnostics.csv", index=False)
if CONFIG["print_metrics"]:
    print("Source-only diagnostics before opening Sketch")
    display(source_frame.round(4))


## 5. Final Sketch evaluation (run only after all Task 3 configurations are fixed)

The next cell is the first permitted Sketch access. It produces final Sketch accuracy/macro-F1, relative changes, and the preconfigured SAM-strength study table.

In [ ]:
# This placement enforces the assignment's unseen-target protocol in the notebook flow.
# This is the first Task 3 code that accesses Sketch. Do not move it above training or source diagnostics.
if not (CONFIG["pacs_root"] / "sketch").is_dir():
    raise FileNotFoundError(
        f"PACS Sketch is missing from Google Drive: {CONFIG['pacs_root'] / 'sketch'}"
    )
sketch_records = []
for y, c in enumerate(CLASSES):
    class_dir = CONFIG["pacs_root"] / "sketch" / c
    for image_path in sorted(class_dir.glob("*")):
        sketch_records.append(
            {"path": str(image_path), "domain": "sketch", "label": y}
        )
sketch = pd.DataFrame(sketch_records)
final = []
sketch_predictions = {}
for row in source_rows:
    m = {"erm": erm, "dan_dg": dan_dg, "sam": sam}[row["method"]]
    result = evaluate(m, sketch)
    sketch_predictions[row["method"]] = result
    final.append(
        {**row, "sketch_accuracy": result["accuracy"], "sketch_macro_f1": result["macro_f1"]}
    )
final = pd.DataFrame(final)
final["sketch_accuracy_change_vs_erm"] = (
    final.sketch_accuracy - final.loc[final.method == "erm", "sketch_accuracy"].iloc[0]
)
final.to_csv(RESULTS / "final_metrics.csv", index=False)


### Sketch class-level diagnostics

After all Task 3 settings are fixed, this cell saves per-class accuracy changes and full confusion counts and displays the final diagnostic tables.

In [ ]:
# Save class-level accuracy changes and full confusions for final failure analysis.
baseline = sketch_predictions["erm"]
per_class_rows = []
confusion_rows = []
for method, result in sketch_predictions.items():
    matrix = confusion_matrix(result["y"], result["pred"], labels=range(7))
    for class_index, class_name in enumerate(CLASSES):
        selected = result["y"] == class_index
        class_accuracy = np.mean(result["pred"][selected] == class_index)
        baseline_selected = baseline["y"] == class_index
        baseline_accuracy = np.mean(baseline["pred"][baseline_selected] == class_index)
        per_class_rows.append(
            {
                "method": method,
                "class": class_name,
                "accuracy": class_accuracy,
                "change_vs_erm": class_accuracy - baseline_accuracy,
            }
        )
        for predicted_index, count in enumerate(matrix[class_index]):
            confusion_rows.append(
                {
                    "method": method,
                    "true_class": class_name,
                    "predicted_class": CLASSES[predicted_index],
                    "count": int(count),
                }
            )
pd.DataFrame(per_class_rows).to_csv(RESULTS / "sketch_per_class.csv", index=False)
pd.DataFrame(confusion_rows).to_csv(RESULTS / "sketch_confusions.csv", index=False)
if CONFIG["print_metrics"]:
    print("Final source and Sketch comparison")
    display(final.round(4))
    print("Sketch accuracy by class and change from ERM")
    display(pd.DataFrame(per_class_rows).round(4))
    confusion_frame = pd.DataFrame(confusion_rows)
    errors = confusion_frame[
        (confusion_frame["true_class"] != confusion_frame["predicted_class"])
        & (confusion_frame["count"] > 0)
    ]
    dominant_errors = (
        errors.sort_values("count", ascending=False)
        .groupby(["method", "true_class"], sort=False)
        .head(1)
    )
    print("Most frequent incorrect prediction for each Sketch class")
    display(dominant_errors.reset_index(drop=True))


### Fixed SAM-radius study

This cell runs the radii specified in the configuration. Source-validation metrics remain separate from Sketch accuracy, which is recorded only for final analysis.

In [ ]:
# Controlled SAM study; settings are fixed before viewing this final-analysis metric.
study = []
for rho in CONFIG["study_rhos"]:
    candidate, _ = train("sam", rho)
    validation_f1 = []
    for domain in SOURCES:
        validation_rows = source[
            (source.domain == domain) & (source.split == "val")
        ]
        validation_f1.append(evaluate(candidate, validation_rows)["macro_f1"])
    study.append(
        {
            "rho": rho,
            "mean_source_f1": np.mean(validation_f1),
            "sharpness_proxy": sharpness(candidate),
            "sketch_accuracy_final_analysis_only": evaluate(candidate, sketch)["accuracy"],
        }
    )
study_frame = pd.DataFrame(study)
study_frame.to_csv(RESULTS / "sam_strength_study.csv", index=False)
if CONFIG["print_metrics"]:
    print("SAM radius study (Sketch values are final analysis only)")
    display(study_frame.round(4))


### Source-training curves

This cell saves per-epoch source losses as CSV, displays the final-epoch table, and displays and saves the classification/MMD figure.

In [ ]:
# Source training histories are saved so both loss terms can be inspected.
pd.DataFrame(dan_history).to_csv(RESULTS / "dan_dg_history.csv", index=False)
pd.DataFrame(sam_history).to_csv(RESULTS / "sam_history.csv", index=False)
if CONFIG["print_metrics"]:
    final_epochs = [
        {"method": "dan_dg", **dan_history[-1]},
        {"method": "sam", **sam_history[-1]},
    ]
    print("Last source-training epoch; full curves are saved to CSV")
    display(pd.DataFrame(final_epochs).round(4))
if CONFIG["plot"]:
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    for method, history in [("DAN-DG", dan_history), ("SAM", sam_history)]:
        axes[0].plot(
            [entry["epoch"] for entry in history],
            [entry["classification_loss"] for entry in history],
            label=method,
        )
    axes[1].plot(
        [entry["epoch"] for entry in dan_history],
        [entry["mmd_penalty"] for entry in dan_history],
        label="DAN-DG",
    )
    axes[0].set(title="Source classification loss", xlabel="epoch", ylabel="cross-entropy")
    axes[1].set(title="Pairwise source MMD", xlabel="epoch", ylabel="MMD")
    axes[0].legend()
    axes[1].legend()
    figure.savefig(REPO_RESULTS_DIR / "source_training_losses.png", dpi=150, bbox_inches="tight")
    plt.show()
